#Name - Pratik Pradeep Rughe

# Option A – Item Type Conversion in the SSH Open Marketplace

The SSH Open Marketplace includes different types of records, such as tools, services, publications, datasets, training materials, and workflows. While these item types share some common metadata, each of them also has its own structure and type-specific fields.

In this notebook, I focus on **Option A: Item type conversion**. I take an existing Publication record from the API and convert it into a draft Dataset record. This reflects a real-world situation where a resource might be submitted under the wrong category, and a curator needs to adjust it to better fit its actual purpose.

Rather than trying to build a fully automated solution, I focus on making the process clear and easy to understand. The goal is to show how fields can be mapped between types, which ones can be reused directly, which ones need some adjustment, and where information is missing or cannot be safely carried over.

## Environment notes

This notebook was written for a standard Python environment. The required packages are listed in `requirements_option_a.txt`.

Required packages:

```text
requests
pandas
```

No API write operation is performed. The notebook only reads public data from the SSH Open Marketplace API and creates a local JSON draft.

## 1. Setup

This section imports the libraries needed to call the API, inspect JSON data, organize the field mapping, and export the converted draft as valid JSON.

In [43]:
import requests  # for API requests
import pandas as pd  # for mapping table and readable tabular output
import json  # for exporting the converted draft as JSON
from pprint import pprint  # for readable JSON previews

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)

## 2. API helper functions

I define a small helper function for calling the SSH Open Marketplace API. Keeping this logic in one place makes the notebook easier to read and reuse.

In [44]:
BASE_URL = "https://marketplace-api.sshopencloud.eu/api"  # base URL for SSH Marketplace API

def fetch_page(endpoint, params=None):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",  # construct full API URL
        params=params or {},  # pass query parameters (pagination, filters)
        timeout=30  # set request timeout
    )
    response.raise_for_status()  # raise error if request fails (e.g., 4xx/5xx)
    return response.json()  # return response as JSON

## 3. Retrieve a Publication item

For this conversion, I use a real Publication record from the API. I fetch a small page of publication records and select the first item that contains enough useful metadata for conversion.

The exact record returned by the API may change over time, so the selection logic is intentionally simple and reproducible.

In [36]:
def fetch_publications(max_pages=3, perpage=20):
    publications = []  # list to store all fetched publication items

    for page in range(1, max_pages + 1):  # loop through pages
        data = fetch_page(
            "publications",  # API endpoint for publications
            params={"page": page, "perpage": perpage}  # pagination parameters
        )

        page_items = data.get("publications", [])  # extract publications list from response

        if not page_items:  # stop if no data returned
            print(f"No publications found on page {page}. Stopping.")
            break

        publications.extend(page_items)  # add items to main list
        print(f"Fetched page {page}: {len(page_items)} publications")  # log progress

    return publications  # return all collected publications


publications = fetch_publications(max_pages=3, perpage=20)  # fetch sample publications

print("Total publications fetched:", len(publications))  # print total count

source_item = publications[0]  # select first item for conversion

print("Selected source item ID:", source_item.get("id"))  # print selected item ID
print("Selected source item label:", source_item.get("label"))  # print selected item title

Fetched page 1: 20 publications
Fetched page 2: 20 publications
Fetched page 3: 20 publications
Total publications fetched: 60
Selected source item ID: 47750
Selected source item label: 3D-ICONS -- 3D Digitisation of Icons of European Architectural and Archaeological Heritage


## 4. Inspect the source item

Before mapping fields, I inspect the source record. This helps me understand which metadata fields are available and which ones may need to be transformed or omitted.

In [37]:
print("Available top-level fields:")  # print all top-level keys in the source item
print(list(source_item.keys()))  # show list of available fields

print("\nSource item preview:")  # separator for readability

preview_fields = [
    "id",
    "category",
    "label",
    "description",
    "persistentId",
    "externalIds",
    "accessibleAt",
    "contributors",
    "properties",
    "source",
    "sourceItemId"
]  # select important fields for preview

source_preview = {
    field: source_item.get(field)  # get value for each selected field
    for field in preview_fields
    if field in source_item  # include only if field exists
}

pprint(source_preview)  # pretty-print selected fields for easy inspection

Available top-level fields:
['id', 'category', 'label', 'persistentId', 'lastInfoUpdate', 'status', 'informationContributor', 'description', 'contributors', 'properties', 'externalIds', 'accessibleAt', 'source', 'sourceItemId', 'relatedItems', 'media']

Source item preview:
{'accessibleAt': ['http://3dicons-project.eu/guidelines-and-case-studies/guidelines'],
 'category': 'publication',
 'contributors': [],
 'description': '3D-ICONS was a pilot project funded under the European '
                "Commission's ICT Policy Support Programme which started on "
                '1st February 2012 and ran for three years. It brought '
                'together partners from across Europe with the relevant '
                'expertise to digitise architectural and archaeological '
                'monuments and buildings in 3D and is designed to: establish a '
                'complete pipeline for the production of 3D replicas of '
                'archaeological monuments and historic buildi

## 5. Field mapping plan

The conversion from Publication to Dataset is not a simple one-to-one copy. Some metadata fields are shared across item types and can be mapped directly, while others require judgement.

I classify each mapping decision as:

- **Direct**: the field can be copied as it is.
- **Transformed**: the field is reused but slightly reshaped or moved.
- **Inferred / defaulted**: the target field needs a value, but the source does not provide enough information.
- **Dropped**: the source information does not clearly belong in the target Dataset draft.

In [38]:
mapping_decisions = pd.DataFrame([
    {
        "source_field": "label",
        "target_field": "label",
        "mapping_type": "Direct",
        "reason": "The publication title can be reused as the draft dataset title."
    },
    {
        "source_field": "description",
        "target_field": "description",
        "mapping_type": "Direct",
        "reason": "The description gives useful context and can describe the dataset draft."
    },
    {
        "source_field": "contributors",
        "target_field": "contributors",
        "mapping_type": "Direct / manual review",
        "reason": "Contributors may be reused, but their exact roles should be checked before submission."
    },
    {
        "source_field": "accessibleAt",
        "target_field": "accessibleAt",
        "mapping_type": "Direct",
        "reason": "Access links are preserved if they point to the resource or landing page."
    },
    {
        "source_field": "externalIds",
        "target_field": "externalIds",
        "mapping_type": "Conservative / manual review",
        "reason": "Publication identifiers may refer to the publication rather than the dataset, so they should be checked manually."
    },
    {
        "source_field": "persistentId",
        "target_field": "derivedFrom.originalPersistentId",
        "mapping_type": "Transformed",
        "reason": "The original publication persistent ID is kept for traceability, not used as the new dataset ID."
    },
    {
        "source_field": "source / sourceItemId",
        "target_field": "source / sourceItemId",
        "mapping_type": "Direct",
        "reason": "Source information is useful for provenance and traceability."
    },
    {
        "source_field": "properties",
        "target_field": "properties",
        "mapping_type": "Partial reuse",
        "reason": "General properties such as language can be reused, but dataset-specific properties still need review."
    },
    {
        "source_field": "publication-specific metadata",
        "target_field": "not mapped",
        "mapping_type": "Dropped",
        "reason": "Fields such as journal, volume, issue, or pages do not directly describe a dataset."
    },
    {
        "source_field": "missing dataset-specific metadata",
        "target_field": "curation.reviewNotes",
        "mapping_type": "Flagged",
        "reason": "Dataset-specific information such as format, spatial coverage, and access conditions cannot be safely inferred."
    },
])

mapping_decisions

,source_field,target_field,mapping_type,reason
0,label,label,Direct,The publication title can be reused as the draft dataset title.
1,description,description,Direct,The description gives useful context and can describe the dataset draft.
2,contributors,contributors,Direct / manual review,"Contributors may be reused, but their exact roles should be checked before submission."
3,accessibleAt,accessibleAt,Direct,Access links are preserved if they point to the resource or landing page.
4,externalIds,externalIds,Conservative / manual review,"Publication identifiers may refer to the publication rather than the dataset, so they should be checked manually."
5,persistentId,derivedFrom.originalPersistentId,Transformed,"The original publication persistent ID is kept for traceability, not used as the new dataset ID."
6,source / sourceItemId,source / sourceItemId,Direct,Source information is useful for provenance and traceability.
7,properties,properties,Partial reuse,"General properties such as language can be reused, but dataset-specific properties still need review."
8,publication-specific metadata,not mapped,Dropped,"Fields such as journal, volume, issue, or pages do not directly describe a dataset."
9,missing dataset-specific metadata,curation.reviewNotes,Flagged,"Dataset-specific information such as format, spatial coverage, and access conditions cannot be safely inferred."


## 6. Conversion helper functions

This section contains small helper functions that make the conversion safer. I avoid inventing metadata that is not available in the source record. When a value cannot be inferred, I either leave it empty or explicitly mark it for curator review.

In [39]:
def safe_copy(item, field, default=None):
    return item.get(field, default)  # safely get field value or return default if missing


def build_conversion_note(source):
    source_id = source.get("id", "unknown")  # get original publication ID
    source_label = source.get("label", "unknown")  # get original title

    return (
        "Draft dataset record generated from a Publication item for review. "
        f"Original publication ID: {source_id}. "  # include original ID for traceability
        f"Original title: {source_label}. "  # include original title
        "Dataset-specific metadata should be checked manually before submission."  # note for manual validation
    )


def build_review_notes(publication):
    notes = [
        "Converted from Publication to Dataset draft.",  # basic conversion note
        "Check whether publication identifiers refer to the dataset itself or only to the publication.",  # caution about IDs
        "Add missing dataset-specific metadata where available."  # reminder to enrich data
    ]

    if not publication.get("contributors"):
        notes.append(
            "No contributors were available in the selected publication record, so the contributors list is empty."
        )  # explain empty contributors

    if not publication.get("externalIds"):
        notes.append(
            "No external identifiers were available in the selected publication record."
        )  # explain empty external IDs

    notes.append(
        "Dataset-specific fields such as format, temporal coverage, spatial coverage, and access conditions were not inferred automatically."
    )  # highlight missing dataset-specific fields

    return notes  # return list of review notes

## 7. Convert Publication to Dataset draft

The function below creates a draft Dataset JSON object. It preserves the fields that can reasonably be reused and adds a provenance block to show where the draft came from.

I intentionally avoid submitting anything back to the API. The output is only a local JSON draft that could be reviewed and adjusted by a curator.

In [40]:
def convert_publication_to_dataset(publication):
    dataset_draft = {
        "category": "dataset",  # change item type to dataset
        "label": safe_copy(publication, "label", ""),  # reuse title
        "description": safe_copy(publication, "description", ""),  # reuse description
        "contributors": safe_copy(publication, "contributors", []),  # copy contributors (may need review)
        "accessibleAt": safe_copy(publication, "accessibleAt", []),  # copy access links
        "externalIds": safe_copy(publication, "externalIds", []),  # copy external IDs (needs validation)
        "source": safe_copy(publication, "source", None),  # preserve source info
        "sourceItemId": safe_copy(publication, "sourceItemId", None),  # preserve source item ID
        "properties": safe_copy(publication, "properties", []),  # reuse general properties
        "conversionQuality": "partial",  # mark conversion as incomplete
        "curation": {
            "status": "draft",  # mark as draft
            "needsManualReview": True,  # flag for human review
            "reviewNotes": build_review_notes(publication)  # attach review notes
        },
        "derivedFrom": {
            "originalCategory": publication.get("category", "publication"),  # store original type
            "originalId": publication.get("id"),  # store original ID
            "originalPersistentId": publication.get("persistentId"),  # store original persistent ID
            "conversionNote": build_conversion_note(publication)  # add traceability note
        }
    }

    return dataset_draft  # return converted dataset draft


converted_dataset = convert_publication_to_dataset(source_item)  # convert selected item

print("Converted draft keys:")
print(list(converted_dataset.keys()))  # display keys of final dataset draft

Converted draft keys:
['category', 'label', 'description', 'contributors', 'accessibleAt', 'externalIds', 'source', 'sourceItemId', 'properties', 'conversionQuality', 'curation', 'derivedFrom']


## 8. Output converted JSON

The converted result is shown as valid JSON. This draft is not submitted to the API, but it follows a clear structure and keeps the original metadata traceable.

In [41]:
converted_json = json.dumps(
    converted_dataset,  # convert Python dict to JSON string
    indent=2,  # pretty-print with indentation
    ensure_ascii=False  # keep Unicode characters readable
)

print(converted_json)  # display the final JSON output

{
  "category": "dataset",
  "label": "3D-ICONS -- 3D Digitisation of Icons of European Architectural and Archaeological Heritage",
  "description": "3D-ICONS was a pilot project funded under the European Commission's ICT Policy Support Programme which started on 1st February 2012 and ran for three years. It brought together partners from across Europe with the relevant expertise to digitise architectural and archaeological monuments and buildings in 3D and is designed to: establish a complete pipeline for the production of 3D replicas of archaeological monuments and historic buildings which covers all technical, legal and organisational aspects; create 3D models and a range of other materials (images, texts and videos) of a series of internationally important monuments and buildings; and contribute content to Europeana using the CARARE aggregation service.",
  "contributors": [],
  "accessibleAt": [
    "http://3dicons-project.eu/guidelines-and-case-studies/guidelines"
  ],
  "externa

## 9. Save the converted draft locally

To make the result reusable, I save the converted dataset draft as a JSON file. This file can be inspected separately or attached alongside the notebook if needed.

In [42]:
output_path = "converted_publication_to_dataset_draft.json"  # define output file name

with open(output_path, "w", encoding="utf-8") as f:  # open file in write mode with UTF-8 encoding
    json.dump(
        converted_dataset,  # data to save
        f,
        indent=2,  # pretty-print JSON
        ensure_ascii=False  # preserve Unicode characters
    )

print(f"Saved converted draft to: {output_path}")  # confirm file save

Saved converted draft to: converted_publication_to_dataset_draft.json


## Mapping interpretation

Some parts of the mapping are quite straightforward. Fields like `label`, `description`, `contributors`, and `accessibleAt`  are part of the common metadata structure, so they can usually be reused directly when creating a Dataset draft.

Other fields need to be handled more carefully. For example, an external identifier attached to a Publication often refers to the publication itself, not necessarily to the underlying dataset. Because of this, I keep the identifier but clearly flag it for manual review. In a similar way, the original `persistentId` is not reused as the dataset identifier. Instead, it is stored separately in a provenance section called `derivedFrom` so that the connection to the original record is still preserved.

There is also some unavoidable information loss. Publication-specific details, such as journal information, page numbers, or citation context, are useful in their original setting but don’t directly apply to a Dataset. Rather than forcing these fields into the new structure, I choose to leave them out and focus on the metadata that makes sense for the target type.

## Limitations and reflection

This conversion is deliberately conservative. I avoid adding dataset-specific details that are not clearly available in the original Publication record. This makes the output safer and more transparent, but it also means the result should be treated as a draft rather than a complete Dataset record.

A curator would still need to review and enrich the converted record before it could be submitted properly.

The main limitations are:

- This conversion is shown using one Publication item, so it may not cover every possible publication structure.
- Some useful metadata may be hidden inside nested properties and would require deeper schema-specific interpretation.
- Identifiers from a Publication may refer to the publication itself, not the dataset behind it.
- Dataset-specific details such as data format, temporal coverage, spatial coverage, or access conditions cannot be reliably inferred from the Publication alone.

Overall, this approach prioritizes clarity and traceability over completeness. It is designed to show what can be safely reused, what needs review, and what should not be guessed.

With more time, I would extend the work by checking the formal schema definitions for both item types, validating the draft against the required Dataset fields, and building a reusable mapping configuration for different source and target item-type combinations.